# Практическая работа №4
## Генеративно-состязательные сети: Vanilla GAN и DCGAN

**Студентка:** Бахорина Ольга Александровна  
**Группа:** R4176c  
**Курс:** 1 курс магистратуры  
**Дисциплина:** «Прикладной искусственный интеллект»

Цель работы — обучить и сравнить Vanilla GAN и DCGAN на CIFAR-10,
проанализировать динамику adversarial loss, визуальное качество и
разнообразие генерации, выполнить latent interpolation и рассчитать
FID и Inception Score по единому протоколу.


### Статус реализации

`RUN_MODE = "full"` предназначен только для итогового запуска на
**Kaggle GPU**. Локальный запуск или Colab допускаются только в
`debug`-режиме для smoke-проверки.

В полном режиме:

- CIFAR-10 делится на 40 000 train, 10 000 development и
  10 000 официальный test;
- обе GAN обучаются по одинаковому протоколу;
- checkpoint сохраняются регулярно;
- лучший checkpoint выбирается только по development FID;
- официальный test не используется при выборе checkpoint;
- финальные FID/IS считаются на 10 000 real и 10 000 fake
  изображениях для каждой модели.


### 1. Введение

Генеративная модель изучает распределение обучающих данных и создаёт новые примеры, похожие на наблюдавшиеся. В GAN две нейронные сети обучаются совместно: генератор синтезирует изображения из случайного латентного вектора, а дискриминатор пытается отличить реальные изображения от сгенерированных.

В работе используется CIFAR-10 — набор из 60 000 цветных изображений размера 32×32, разделённых на 10 классов: 50 000 изображений входят в training split и 10 000 — в test split.

### 2. Цель и задачи

**Цель работы:** изучить принципы обучения GAN и сравнить полносвязную и свёрточную архитектуры при генерации изображений CIFAR-10.

Для достижения цели необходимо:

1. загрузить CIFAR-10 и привести изображения к диапазону `[-1, 1]`;
2. реализовать Vanilla GAN и DCGAN на PyTorch;
3. обучить обе модели в сопоставимых условиях и записать `loss_G` и `loss_D`;
4. визуализировать сгенерированные изображения и интерполяцию в латентном пространстве;
5. вычислить FID и Inception Score по фактическим результатам запуска;
6. сравнить качество, устойчивость и вычислительную стоимость моделей.

### 3. Теория GAN

Генератор $G$ преобразует случайный вектор $z \sim p_z$ в изображение $G(z)$. Дискриминатор $D$ оценивает, принадлежит ли вход реальному распределению данных. Классическая минимаксная постановка имеет вид:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1-D(G(z)))].$$

На практике дискриминатор возвращает logit, а бинарная кросс-энтропия вычисляется через `BCEWithLogitsLoss`, которая численно устойчиво объединяет sigmoid и BCE. Поэтому отдельный sigmoid в последнем слое дискриминатора не нужен. Генератор завершает работу функцией `tanh`, согласованной с нормализацией изображений в диапазон `[-1, 1]`.

Vanilla GAN использует полносвязные слои и не учитывает пространственную структуру изображения. DCGAN применяет свёрточные операции, поэтому способна изучать локальные признаки и пространственные зависимости. Значения adversarial loss описывают динамику игры двух сетей, но сами по себе не являются прямой мерой визуального качества. Для количественного сравнения далее будут использованы FID (меньше — лучше) и Inception Score (больше — лучше).

### 4. Описание CIFAR-10

CIFAR-10 содержит 60 000 RGB-изображений 32×32 из десяти классов.

Официальные 50 000 training images стратифицированно разделяются с
`seed=42` на:

- **40 000 train** — обучение GAN;
- **10 000 development** — выбор checkpoint.

Официальные **10 000 test** не участвуют ни в обучении, ни в выборе
checkpoint и используются только для финальной оценки.

Все изображения нормализуются из `[0, 1]` в `[-1, 1]`, что
соответствует `tanh` на выходе генератора.


### 5. Подготовка среды

Полный эксперимент должен выполняться на Kaggle GPU.
`debug`-режим предназначен только для smoke-проверки.

Фиксируются версии Python, CUDA, PyTorch, torchvision,
`torch-fidelity`, параметры эксперимента, seed и путь артефактов.


In [ ]:
%pip install -q torch-fidelity==0.4.0

In [ ]:
import hashlib
import json
import os
import platform
import random
import sys
import tempfile
import time
from datetime import datetime, timezone
from importlib.metadata import (
    PackageNotFoundError,
    version as package_version,
)
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch_fidelity
from IPython.display import Markdown, display
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

RUN_MODE = "full"
VALID_RUN_MODES = {"debug", "full"}

if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(
        "RUN_MODE должен быть одним из "
        f"{sorted(VALID_RUN_MODES)}"
    )

SEED = 42
FAST_MODE = RUN_MODE == "debug"

LATENT_DIM = 128
IMAGE_SHAPE = (3, 32, 32)
BATCH_SIZE = 64
LEARNING_RATE = 2e-4
ADAM_BETAS = (0.5, 0.999)

FIXED_NOISE_COUNT = 64
FAST_EPOCHS = 1
FULL_EPOCHS = 100
FAST_SUBSET_SIZE = 2_048

TRAIN_IMAGES = 40_000
DEVELOPMENT_IMAGES = 10_000
TEST_IMAGES = 10_000

FAST_EVAL_IMAGES = 1_000
FULL_EVAL_IMAGES = 10_000
DEVELOPMENT_EVAL_IMAGES = 10_000

CHECKPOINT_INTERVAL = 25
IS_SPLITS = 10

NUM_WORKERS = (
    2
    if torch.cuda.is_available()
    else 0
)

KAGGLE_ROOT = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_ROOT.is_dir()

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if RUN_MODE == "full":
    if not IS_KAGGLE:
        raise RuntimeError(
            "Финальный full-run должен выполняться "
            "в Kaggle."
        )

    if DEVICE.type != "cuda":
        raise RuntimeError(
            "Для финального full-run требуется "
            "Kaggle GPU."
        )

if IS_KAGGLE:
    PROJECT_ROOT = KAGGLE_ROOT
else:
    PROJECT_ROOT = (
        Path.cwd().parent
        if Path.cwd().name == "notebooks"
        else Path.cwd()
    )

DATA_ROOT = PROJECT_ROOT / "data"
RUNTIME_ROOT = PROJECT_ROOT / "lab4_gan_runtime"
ARTIFACT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "lab4"
    / RUN_MODE
)
FIGURE_ROOT = ARTIFACT_ROOT / "figures"
TABLE_ROOT = ARTIFACT_ROOT / "tables"

EPOCHS = (
    FAST_EPOCHS
    if FAST_MODE
    else FULL_EPOCHS
)

RUN_ID = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%dT%H%M%SZ")
    + "_p4_gan"
)


def installed_version(
    distribution_name,
):
    try:
        return package_version(
            distribution_name
        )
    except PackageNotFoundError:
        return "not-installed"


def validate_experiment_configuration():
    if RUN_MODE != "full":
        return

    expected = {
        "epochs": 100,
        "batch_size": 64,
        "test_images": 10_000,
        "development_images": 10_000,
    }

    actual = {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "test_images": FULL_EVAL_IMAGES,
        "development_images": (
            DEVELOPMENT_EVAL_IMAGES
        ),
    }

    mismatches = {
        key: (
            expected[key],
            actual[key],
        )
        for key in expected
        if expected[key]
        != actual[key]
    }

    if mismatches:
        raise RuntimeError(
            "Некорректная FULL-конфигурация: "
            f"{mismatches}"
        )


validate_experiment_configuration()

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
FIGURE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
TABLE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_MANIFEST_PATH = (
    ARTIFACT_ROOT
    / "run_manifest.json"
)

RUN_MANIFEST = {
    "run_id": RUN_ID,
    "created_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "run_mode": RUN_MODE,
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "latent_dim": LATENT_DIM,
    "learning_rate": LEARNING_RATE,
    "adam_betas": list(ADAM_BETAS),
    "checkpoint_interval": (
        CHECKPOINT_INTERVAL
    ),
    "train_images": TRAIN_IMAGES,
    "development_images": (
        DEVELOPMENT_IMAGES
    ),
    "test_images": TEST_IMAGES,
    "evaluation_images": (
        FAST_EVAL_IMAGES
        if FAST_MODE
        else FULL_EVAL_IMAGES
    ),
    "is_splits": IS_SPLITS,
    "device": str(DEVICE),
    "is_kaggle": IS_KAGGLE,
    "cuda_available": (
        torch.cuda.is_available()
    ),
    "cuda_version": (
        torch.version.cuda
    ),
    "cudnn_version": (
        torch.backends.cudnn.version()
    ),
    "python": (
        sys.version.split()[0]
    ),
    "platform": (
        platform.platform()
    ),
    "artifact_root": (
        str(ARTIFACT_ROOT)
    ),
    "packages": {
        name: installed_version(name)
        for name in (
            "torch",
            "torchvision",
            "torch-fidelity",
            "numpy",
            "pandas",
            "matplotlib",
            "seaborn",
            "Pillow",
        )
    },
}

RUN_MANIFEST_PATH.write_text(
    json.dumps(
        RUN_MANIFEST,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("PyTorch:", torch.__version__)
print("Устройство:", DEVICE)
print("Kaggle:", IS_KAGGLE)
print("Режим:", RUN_MODE.upper())
print("Эпох:", EPOCHS)
print("Run ID:", RUN_ID)
print("Артефакты:", ARTIFACT_ROOT)


### 6. Загрузка и предобработка CIFAR-10

Для воспроизводимости фиксируется `seed=42`.

В полном режиме официальный training split CIFAR-10
стратифицированно делится на 40 000 train и 10 000 development.
Официальные 10 000 test остаются полностью независимыми.

Обе архитектуры используют одинаковые train/development/test
manifests.


In [ ]:
def set_reproducible_seed(
    seed=SEED,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cifar10_transform():
    return transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5),
            ),
        ]
    )


def stratified_train_dev_indices(
    targets,
    seed=SEED,
):
    targets = np.asarray(
        targets
    )
    rng = np.random.default_rng(
        seed
    )

    train_indices = []
    development_indices = []

    for class_id in sorted(
        np.unique(targets)
    ):
        class_indices = np.flatnonzero(
            targets == class_id
        )
        rng.shuffle(
            class_indices
        )

        development_indices.extend(
            class_indices[:1_000]
            .tolist()
        )
        train_indices.extend(
            class_indices[1_000:]
            .tolist()
        )

    train_indices = np.asarray(
        train_indices,
        dtype=np.int64,
    )
    development_indices = np.asarray(
        development_indices,
        dtype=np.int64,
    )

    rng.shuffle(
        train_indices
    )
    rng.shuffle(
        development_indices
    )

    return (
        train_indices,
        development_indices,
    )


def select_subset_indices(
    indices,
    subset_size,
    seed=SEED,
):
    if subset_size < 1:
        raise ValueError(
            "subset_size должен быть "
            "положительным"
        )

    indices = np.asarray(
        indices
    )
    sample_size = min(
        len(indices),
        subset_size,
    )

    rng = np.random.default_rng(
        seed
    )

    return rng.choice(
        indices,
        size=sample_size,
        replace=False,
    )


def denormalize_images(
    images,
):
    return (
        images.mul(0.5)
        .add(0.5)
        .clamp(0.0, 1.0)
    )


def create_cifar10_loaders(
    data_root=DATA_ROOT,
    fast_mode=FAST_MODE,
    download=True,
):
    transform = cifar10_transform()

    train_full = datasets.CIFAR10(
        root=data_root,
        train=True,
        download=download,
        transform=transform,
    )

    test_dataset = datasets.CIFAR10(
        root=data_root,
        train=False,
        download=download,
        transform=transform,
    )

    (
        train_indices,
        development_indices,
    ) = stratified_train_dev_indices(
        train_full.targets,
        seed=SEED,
    )

    if fast_mode:
        active_train_indices = (
            select_subset_indices(
                train_indices,
                FAST_SUBSET_SIZE,
                seed=SEED,
            )
        )
    else:
        active_train_indices = (
            train_indices
        )

    train_dataset = Subset(
        train_full,
        active_train_indices,
    )
    development_dataset = Subset(
        train_full,
        development_indices,
    )

    loader_generator = (
        torch.Generator()
        .manual_seed(SEED)
    )

    common_loader_options = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": (
            DEVICE.type == "cuda"
        ),
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=loader_generator,
        **common_loader_options,
    )

    development_loader = DataLoader(
        development_dataset,
        shuffle=False,
        **common_loader_options,
    )

    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        **common_loader_options,
    )

    return (
        train_full,
        train_dataset,
        development_dataset,
        test_dataset,
        train_loader,
        development_loader,
        test_loader,
        train_indices,
        development_indices,
    )


set_reproducible_seed()


In [ ]:
DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

(
    train_full_dataset,
    train_dataset,
    development_dataset,
    test_dataset,
    train_loader,
    development_loader,
    test_loader,
    train_indices,
    development_indices,
) = create_cifar10_loaders()

assert len(train_full_dataset) == 50_000
assert len(development_dataset) == 10_000
assert len(test_dataset) == 10_000

if FAST_MODE:
    assert len(train_dataset) == min(
        FAST_SUBSET_SIZE,
        TRAIN_IMAGES,
    )
else:
    assert len(train_dataset) == 40_000

train_targets = np.asarray(
    train_full_dataset.targets
)

development_class_counts = np.bincount(
    train_targets[
        development_indices
    ],
    minlength=10,
)

train_class_counts = np.bincount(
    train_targets[
        train_indices
    ],
    minlength=10,
)

assert np.all(
    development_class_counts
    == 1_000
)
assert np.all(
    train_class_counts
    == 4_000
)

split_manifest = pd.DataFrame(
    {
        "split": [
            "train",
            "development",
            "test",
        ],
        "images": [
            40_000,
            10_000,
            10_000,
        ],
    }
)

split_manifest.to_csv(
    TABLE_ROOT
    / "cifar10_split_manifest.csv",
    index=False,
)

np.savez_compressed(
    TABLE_ROOT
    / "cifar10_split_indices.npz",
    train_indices=train_indices,
    development_indices=(
        development_indices
    ),
)

real_batch, real_labels = next(
    iter(train_loader)
)

batch_min = float(
    real_batch.min()
)
batch_max = float(
    real_batch.max()
)

assert real_batch.ndim == 4
assert tuple(
    real_batch.shape[1:]
) == (3, 32, 32)
assert (
    -1.0001
    <= batch_min
    <= batch_max
    <= 1.0001
)

display(split_manifest)

print(
    "Training images:",
    len(train_dataset),
)
print(
    "Development images:",
    len(development_dataset),
)
print(
    "Test images:",
    len(test_dataset),
)
print(
    "Batch shape:",
    tuple(real_batch.shape),
)
print(
    "Normalized range:",
    f"[{batch_min:.3f}, "
    f"{batch_max:.3f}]",
)


### 7. Анализ исходных изображений

Хотя class labels не используются при обучении GAN, распределение классов помогает проверить состав набора. Статистика строится по полному training split, даже когда обучение выполняется в быстром режиме.

In [ ]:
class_counts = np.bincount(
    np.asarray(train_full_dataset.targets), minlength=len(train_full_dataset.classes)
)
class_distribution = pd.DataFrame(
    {
        "class": train_full_dataset.classes,
        "images": class_counts,
    }
)
display(class_distribution)
class_distribution.to_csv(
    TABLE_ROOT / "cifar10_class_distribution.csv", index=False
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(
    data=class_distribution, x="class", y="images", color="steelblue", ax=ax
)
ax.set_title("Распределение классов CIFAR-10: training split")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
fig.savefig(
    FIGURE_ROOT / "cifar10_class_distribution.png", dpi=180, bbox_inches="tight"
)
plt.show()

In [ ]:
preview_count = min(16, len(real_batch))
preview_generator = torch.Generator().manual_seed(SEED)
preview_indices = torch.randperm(
    len(real_batch),
    generator=preview_generator,
)[:preview_count]
preview_images = denormalize_images(real_batch[preview_indices])
preview_labels = real_labels[preview_indices]
grid = make_grid(preview_images, nrow=4, padding=2)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(grid.permute(1, 2, 0).cpu())
ax.set_title("Реальные изображения CIFAR-10 после обратной нормализации")
ax.axis("off")
fig.tight_layout()
fig.savefig(
    FIGURE_ROOT / "cifar10_real_images.png", dpi=180, bbox_inches="tight"
)
plt.show()

print(
    "Классы изображений в сетке:",
    [train_full_dataset.classes[int(label)] for label in preview_labels],
)

После обратной нормализации значения снова находятся в `[0, 1]`, поэтому цвета отображаются корректно. Низкое разрешение 32×32 и разнообразие объектов делают CIFAR-10 содержательной проверкой способности GAN одновременно изучать локальные текстуры и глобальную структуру. Равномерное распределение исходных классов не гарантирует, что безусловная GAN воспроизведёт их одинаково часто; разнообразие сэмплов будет анализироваться после обучения.

### 8. Архитектура Vanilla GAN

Vanilla GAN обрабатывает изображение как вектор длины $3 \cdot 32 \cdot 32 = 3072$. Генератор получает $z \sim \mathcal{N}(0, I)$ размерности 128 и последовательно увеличивает размер признакового представления полносвязными слоями. Финальный `Tanh` ограничивает выход диапазоном `[-1, 1]`.

| Компонент | Последовательность размерностей | Активации |
|---|---|---|
| Generator | 128 → 256 → 512 → 1024 → 3072 | ReLU, Tanh |
| Discriminator | 3072 → 1024 → 512 → 1 | LeakyReLU(0.2), logit |

Дискриминатор не содержит `Sigmoid`: `BCEWithLogitsLoss` объединяет sigmoid и бинарную кросс-энтропию в одной численно устойчивой операции.

In [ ]:
class VanillaGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, image_shape=IMAGE_SHAPE):
        super().__init__()
        self.image_shape = tuple(image_shape)
        output_features = int(np.prod(self.image_shape))
        self.network = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, output_features),
            nn.Tanh(),
        )

    def forward(self, latent_vectors):
        generated = self.network(latent_vectors)
        return generated.view(latent_vectors.size(0), *self.image_shape)


class VanillaDiscriminator(nn.Module):
    def __init__(self, image_shape=IMAGE_SHAPE):
        super().__init__()
        input_features = int(np.prod(image_shape))
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_features, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 1),
        )

    def forward(self, images):
        return self.network(images)


def count_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

In [ ]:
vanilla_generator = VanillaGenerator().to(DEVICE)
vanilla_discriminator = VanillaDiscriminator().to(DEVICE)
adversarial_loss = nn.BCEWithLogitsLoss()
vanilla_optimizer_G = torch.optim.Adam(
    vanilla_generator.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS
)
vanilla_optimizer_D = torch.optim.Adam(
    vanilla_discriminator.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS
)

fixed_noise_2d = torch.randn(
    FIXED_NOISE_COUNT, LATENT_DIM, generator=torch.Generator().manual_seed(SEED)
).to(DEVICE)
smoke_noise = fixed_noise_2d[:4]
vanilla_generator.eval()
vanilla_discriminator.eval()
with torch.no_grad():
    smoke_images = vanilla_generator(smoke_noise)
    smoke_logits = vanilla_discriminator(smoke_images)
vanilla_generator.train()
vanilla_discriminator.train()

assert tuple(smoke_images.shape) == (4, 3, 32, 32)
assert tuple(smoke_logits.shape) == (4, 1)
assert torch.isfinite(smoke_images).all()
assert torch.isfinite(smoke_logits).all()
assert float(smoke_images.min()) >= -1.0001
assert float(smoke_images.max()) <= 1.0001

vanilla_architecture_summary = pd.DataFrame(
    [
        {"component": "Generator", "parameters": count_parameters(vanilla_generator)},
        {
            "component": "Discriminator",
            "parameters": count_parameters(
                vanilla_discriminator
            ),
        },
    ]
)
vanilla_architecture_summary.to_csv(
    TABLE_ROOT / "vanilla_architecture.csv", index=False
)
display(vanilla_architecture_summary.style.format({"parameters": "{:,}"}))
print("Generator output:", tuple(smoke_images.shape))
print("Discriminator output:", tuple(smoke_logits.shape))

Smoke-check подтверждает совместимость размерностей: генератор создаёт batch RGB-изображений 32×32, а дискриминатор возвращает ровно один logit на изображение. Число параметров вычисляется из фактических моделей. Полносвязный Generator особенно затратен в последнем слое `1024 → 3072`; это одна из причин сравнить его со свёрточной DCGAN.

### 9. Обучение Vanilla GAN

На каждом batch сначала обновляется дискриминатор по real и
`detach()`-копии fake batch, затем обновляется генератор.

`loss_D` и `loss_G` логируются для каждого шага.

Каждые 25 эпох сохраняются:

- Generator и Discriminator;
- состояния обоих Adam optimizer;
- номер эпохи и история loss;
- состояния генераторов случайных чисел;
- fixed-noise snapshot.

После завершения обучения лучший checkpoint выбирается только по
development FID.


In [ ]:
def set_requires_grad(
    model,
    requires_grad,
):
    for parameter in model.parameters():
        parameter.requires_grad_(
            requires_grad
        )


def train_gan_batch(
    generator,
    discriminator,
    real_images,
    optimizer_G,
    optimizer_D,
    loss_function,
    latent_shape,
    device=DEVICE,
):
    generator.train()
    discriminator.train()

    real_images = real_images.to(
        device,
        non_blocking=True,
    )
    batch_size = real_images.size(0)

    real_targets = torch.ones(
        batch_size,
        1,
        device=device,
    )
    fake_targets = torch.zeros(
        batch_size,
        1,
        device=device,
    )

    set_requires_grad(
        discriminator,
        True,
    )
    optimizer_D.zero_grad(
        set_to_none=True
    )

    real_logits = discriminator(
        real_images
    )
    discriminator_real_loss = (
        loss_function(
            real_logits,
            real_targets,
        )
    )

    discriminator_noise = torch.randn(
        batch_size,
        *latent_shape,
        device=device,
    )
    discriminator_fakes = generator(
        discriminator_noise
    )
    fake_logits = discriminator(
        discriminator_fakes.detach()
    )
    discriminator_fake_loss = (
        loss_function(
            fake_logits,
            fake_targets,
        )
    )

    discriminator_loss = 0.5 * (
        discriminator_real_loss
        + discriminator_fake_loss
    )
    discriminator_loss.backward()
    optimizer_D.step()

    set_requires_grad(
        discriminator,
        False,
    )
    optimizer_G.zero_grad(
        set_to_none=True
    )

    generator_noise = torch.randn(
        batch_size,
        *latent_shape,
        device=device,
    )
    generated_images = generator(
        generator_noise
    )
    generator_logits = discriminator(
        generated_images
    )
    generator_loss = loss_function(
        generator_logits,
        real_targets,
    )
    generator_loss.backward()
    optimizer_G.step()

    set_requires_grad(
        discriminator,
        True,
    )

    return {
        "loss_D": float(
            discriminator_loss.detach()
        ),
        "loss_G": float(
            generator_loss.detach()
        ),
    }


def generate_fixed_samples(
    generator,
    fixed_noise,
):
    was_training = generator.training
    generator.eval()

    try:
        with torch.no_grad():
            return generator(
                fixed_noise
            ).detach().cpu()
    finally:
        generator.train(
            was_training
        )


def save_training_checkpoint(
    path,
    generator,
    discriminator,
    optimizer_G,
    optimizer_D,
    epoch,
    latent_shape,
    history,
    model_name,
):
    checkpoint = {
        "run_id": RUN_ID,
        "model_name": model_name,
        "epoch": epoch,
        "generator_state_dict": (
            generator.state_dict()
        ),
        "discriminator_state_dict": (
            discriminator.state_dict()
        ),
        "optimizer_G_state_dict": (
            optimizer_G.state_dict()
        ),
        "optimizer_D_state_dict": (
            optimizer_D.state_dict()
        ),
        "latent_shape": tuple(
            latent_shape
        ),
        "history": history,
        "seed": SEED,
        "python_random_state": (
            random.getstate()
        ),
        "numpy_random_state": (
            np.random.get_state()
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
    }

    if torch.cuda.is_available():
        checkpoint[
            "cuda_rng_state_all"
        ] = (
            torch.cuda
            .get_rng_state_all()
        )

    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    torch.save(
        checkpoint,
        path,
    )

    return path


def train_gan(
    generator,
    discriminator,
    data_loader,
    optimizer_G,
    optimizer_D,
    loss_function,
    latent_shape,
    fixed_noise,
    epochs,
    model_name,
    checkpoint_path,
    checkpoint_interval=(
        CHECKPOINT_INTERVAL
    ),
    device=DEVICE,
):
    history = {
        "step": [],
        "epoch": [],
        "loss_D": [],
        "loss_G": [],
    }

    epoch_times = []
    snapshots = {
        0: generate_fixed_samples(
            generator,
            fixed_noise,
        )
    }
    checkpoint_paths = []

    snapshot_epochs = set(
        range(
            checkpoint_interval,
            epochs + 1,
            checkpoint_interval,
        )
    )
    snapshot_epochs.add(
        epochs
    )

    global_step = 0
    training_started = (
        time.perf_counter()
    )

    checkpoint_path = Path(
        checkpoint_path
    )

    for epoch in range(
        1,
        epochs + 1,
    ):
        epoch_started = (
            time.perf_counter()
        )

        for real_images, _ in data_loader:
            losses = train_gan_batch(
                generator,
                discriminator,
                real_images,
                optimizer_G,
                optimizer_D,
                loss_function,
                latent_shape,
                device,
            )

            global_step += 1

            history["step"].append(
                global_step
            )
            history["epoch"].append(
                epoch
            )
            history["loss_D"].append(
                losses["loss_D"]
            )
            history["loss_G"].append(
                losses["loss_G"]
            )

        epoch_time = (
            time.perf_counter()
            - epoch_started
        )
        epoch_times.append(
            epoch_time
        )

        if epoch in snapshot_epochs:
            snapshots[epoch] = (
                generate_fixed_samples(
                    generator,
                    fixed_noise,
                )
            )

            periodic_path = (
                checkpoint_path
                .with_name(
                    checkpoint_path.stem
                    + f"_epoch_{epoch:03d}"
                    + checkpoint_path.suffix
                )
            )

            checkpoint_paths.append(
                save_training_checkpoint(
                    periodic_path,
                    generator,
                    discriminator,
                    optimizer_G,
                    optimizer_D,
                    epoch,
                    latent_shape,
                    history,
                    model_name,
                )
            )

        print(
            f"{model_name} | "
            f"epoch {epoch:03d}/{epochs:03d} | "
            f"loss_D={losses['loss_D']:.4f} | "
            f"loss_G={losses['loss_G']:.4f} | "
            f"{epoch_time:.1f} s"
        )

    training_time = (
        time.perf_counter()
        - training_started
    )

    final_checkpoint = (
        save_training_checkpoint(
            checkpoint_path,
            generator,
            discriminator,
            optimizer_G,
            optimizer_D,
            epochs,
            latent_shape,
            history,
            model_name,
        )
    )

    return {
        "history": pd.DataFrame(
            history
        ),
        "snapshots": snapshots,
        "training_time_s": (
            training_time
        ),
        "epoch_times_s": (
            epoch_times
        ),
        "checkpoint_path": (
            final_checkpoint
        ),
        "checkpoint_paths": (
            checkpoint_paths
        ),
    }


In [ ]:
set_reproducible_seed()
train_loader.generator.manual_seed(SEED)
vanilla_generator = VanillaGenerator().to(DEVICE)
vanilla_discriminator = VanillaDiscriminator().to(DEVICE)
vanilla_optimizer_G = torch.optim.Adam(
    vanilla_generator.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS
)
vanilla_optimizer_D = torch.optim.Adam(
    vanilla_discriminator.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS
)
VANILLA_CHECKPOINT = RUNTIME_ROOT / "checkpoints" / "vanilla_gan.pt"

vanilla_results = train_gan(
    generator=vanilla_generator,
    discriminator=vanilla_discriminator,
    data_loader=train_loader,
    optimizer_G=vanilla_optimizer_G,
    optimizer_D=vanilla_optimizer_D,
    loss_function=adversarial_loss,
    latent_shape=(LATENT_DIM,),
    fixed_noise=fixed_noise_2d,
    epochs=EPOCHS,
    model_name="Vanilla GAN",
    checkpoint_path=VANILLA_CHECKPOINT,
)

if FAST_MODE:
    print("FAST_MODE: результаты предназначены только для smoke/debug проверки.")

### 10. Результаты Vanilla GAN

Кривые ниже показывают динамику состязательного обучения, но меньший GAN loss не означает автоматически более качественные изображения. Сетки из одинаковых latent vectors позволяют визуально оценить прогресс, разнообразие и возможные признаки mode collapse. Конкретные выводы следует делать только по графикам и изображениям текущего запуска.

In [ ]:
def save_figure(fig, output_path, dpi=180):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    return output_path


def save_tensor_grid(images, output_path, title, nrow=8):
    grid = make_grid(denormalize_images(images), nrow=nrow, padding=2)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    saved_path = save_figure(fig, output_path)
    plt.close(fig)
    return saved_path


def plot_training_history(history, title, output_path):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history["step"], history["loss_D"], label="loss_D", alpha=0.85)
    ax.plot(history["step"], history["loss_G"], label="loss_G", alpha=0.85)
    ax.set(title=title, xlabel="Training step", ylabel="BCE loss")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_figure(fig, output_path)
    plt.show()


def plot_sample_snapshots(
    snapshots, title, output_path, snapshot_dir, nrow=8
):
    ordered_snapshots = sorted(snapshots.items())
    fig, axes = plt.subplots(
        1,
        len(ordered_snapshots),
        figsize=(
            6 * len(ordered_snapshots),
            6,
        ),
        squeeze=False,
    )
    for axis, (epoch, images) in zip(axes[0], ordered_snapshots):
        grid = make_grid(denormalize_images(images), nrow=nrow, padding=2)
        axis.imshow(grid.permute(1, 2, 0).numpy())
        axis.set_title(f"{title}: epoch {epoch}")
        axis.axis("off")
        save_tensor_grid(
            images,
            Path(snapshot_dir) / f"epoch_{epoch:03d}.png",
            f"{title}: epoch {epoch}",
            nrow=nrow,
        )
    fig.tight_layout()
    save_figure(fig, output_path)
    plt.show()

In [ ]:
vanilla_history = vanilla_results["history"]
vanilla_history.to_csv(TABLE_ROOT / "vanilla_training_history.csv", index=False)
plot_training_history(
    vanilla_history,
    "Vanilla GAN: training dynamics",
    FIGURE_ROOT / "vanilla_training_loss.png",
)
plot_sample_snapshots(
    vanilla_results["snapshots"],
    "Vanilla GAN",
    FIGURE_ROOT / "vanilla_snapshots.png",
    FIGURE_ROOT / "vanilla_snapshots",
)

vanilla_mean_epoch_time = float(np.mean(vanilla_results["epoch_times_s"]))
print(f"Training time: {vanilla_results['training_time_s']:.1f} s")
print(f"Mean epoch time: {vanilla_mean_epoch_time:.1f} s")
print(f"Checkpoint: {vanilla_results['checkpoint_path']}")

### 11. Архитектура DCGAN

DCGAN сохраняет пространственную структуру изображений. Генератор принимает тот же latent vector, представленный как тензор `N×128×1×1`, и четырьмя `ConvTranspose2d`-блоками увеличивает разрешение до 32×32. Дискриминатор выполняет обратное понижение разрешения с помощью `Conv2d`.

| Компонент | Размерности | Основные операции |
|---|---|---|
| Generator | 128×1×1 → 256×4×4 → 128×8×8 → 64×16×16 → 3×32×32 | ConvTranspose2d, BatchNorm2d, ReLU, Tanh |
| Discriminator | 3×32×32 → 64×16×16 → 128×8×8 → 256×4×4 → 1×1×1 | Conv2d, BatchNorm2d, LeakyReLU(0.2), logit |

Свёрточные веса инициализируются из $\mathcal{N}(0, 0.02)$, а scale-параметры BatchNorm — из $\mathcal{N}(1, 0.02)$ с нулевым bias. Это стандартная DCGAN initialization, которая задаёт контролируемый начальный масштаб активаций. Отдельный `Sigmoid` в дискриминаторе по-прежнему не используется.

In [ ]:
class DCGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.network = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, latent_vectors):
        return self.network(latent_vectors)


class DCDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0, bias=False),
            nn.Flatten(start_dim=1),
        )

    def forward(self, images):
        return self.network(images)


def initialize_dcgan_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight, mean=1.0, std=0.02)
        nn.init.constant_(module.bias, 0.0)

In [ ]:
dc_generator = DCGenerator().to(DEVICE)
dc_discriminator = DCDiscriminator().to(DEVICE)
dc_generator.apply(initialize_dcgan_weights)
dc_discriminator.apply(initialize_dcgan_weights)

fixed_noise_4d = fixed_noise_2d.view(FIXED_NOISE_COUNT, LATENT_DIM, 1, 1)
dc_generator.eval()
dc_discriminator.eval()
with torch.no_grad():
    dc_smoke_images = dc_generator(fixed_noise_4d[:4])
    dc_smoke_logits = dc_discriminator(dc_smoke_images)
dc_generator.train()
dc_discriminator.train()

assert tuple(dc_smoke_images.shape) == (4, 3, 32, 32)
assert tuple(dc_smoke_logits.shape) == (4, 1)
assert torch.isfinite(dc_smoke_images).all()
assert torch.isfinite(dc_smoke_logits).all()
assert float(dc_smoke_images.min()) >= -1.0001
assert float(dc_smoke_images.max()) <= 1.0001

dcgan_architecture_summary = pd.DataFrame(
    [
        {"component": "Generator", "parameters": count_parameters(dc_generator)},
        {
            "component": "Discriminator",
            "parameters": count_parameters(
                dc_discriminator
            ),
        },
    ]
)
dcgan_architecture_summary.to_csv(
    TABLE_ROOT / "dcgan_architecture.csv", index=False
)
display(dcgan_architecture_summary.style.format({"parameters": "{:,}"}))
print("DCGenerator output:", tuple(dc_smoke_images.shape))
print("DCDiscriminator output:", tuple(dc_smoke_logits.shape))

Проверка размерностей подтверждает, что DCGenerator и DCDiscriminator напрямую совместимы с CIFAR-10 и общим training pipeline. Одни и те же значения fixed noise используются в формах `N×128` для Vanilla GAN и `N×128×1×1` для DCGAN; различается только представление latent vectors. Число параметров обеих свёрточных сетей вычислено из runtime-моделей.

### 12. Обучение DCGAN

DCGAN обучается тем же `train_gan` pipeline, `BCEWithLogitsLoss`, числом эпох, batch size и порядком training batches, что и Vanilla GAN. Для обеих сетей используется Adam с `lr=2e-4`, $\beta_1=0.5$ и $\beta_2=0.999$. Отличаются только архитектура и форма latent input: `(N, 128, 1, 1)` для DCGAN.

Перед обучением модели создаются заново, получают стандартную DCGAN initialization, а DataLoader generator возвращается к `seed=42`. Checkpoint сохраняется в игнорируемый runtime-каталог. Как и ранее, режим `debug` подтверждает работоспособность, но не используется для финальных выводов.

In [ ]:
set_reproducible_seed()
train_loader.generator.manual_seed(SEED)
dc_generator = DCGenerator().to(DEVICE)
dc_discriminator = DCDiscriminator().to(DEVICE)
dc_generator.apply(initialize_dcgan_weights)
dc_discriminator.apply(initialize_dcgan_weights)
dc_optimizer_G = torch.optim.Adam(
    dc_generator.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS
)
dc_optimizer_D = torch.optim.Adam(
    dc_discriminator.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS
)
DCGAN_CHECKPOINT = RUNTIME_ROOT / "checkpoints" / "dcgan.pt"

dcgan_results = train_gan(
    generator=dc_generator,
    discriminator=dc_discriminator,
    data_loader=train_loader,
    optimizer_G=dc_optimizer_G,
    optimizer_D=dc_optimizer_D,
    loss_function=adversarial_loss,
    latent_shape=(LATENT_DIM, 1, 1),
    fixed_noise=fixed_noise_4d,
    epochs=EPOCHS,
    model_name="DCGAN",
    checkpoint_path=DCGAN_CHECKPOINT,
)

if FAST_MODE:
    print("FAST_MODE: результаты DCGAN предназначены только для smoke/debug проверки.")

### 13. Результаты DCGAN

Графики и snapshots формируются из фактического текущего запуска. При анализе важно рассматривать совместно динамику обеих loss, визуальную чёткость и разнообразие fixed-noise samples. Осцилляции loss ожидаемы для состязательного обучения и не позволяют сами по себе объявить одну архитектуру лучшей.

In [ ]:
dcgan_history = dcgan_results["history"]
dcgan_history.to_csv(TABLE_ROOT / "dcgan_training_history.csv", index=False)
plot_training_history(
    dcgan_history,
    "DCGAN: training dynamics",
    FIGURE_ROOT / "dcgan_training_loss.png",
)
plot_sample_snapshots(
    dcgan_results["snapshots"],
    "DCGAN",
    FIGURE_ROOT / "dcgan_snapshots.png",
    FIGURE_ROOT / "dcgan_snapshots",
)

dcgan_mean_epoch_time = float(np.mean(dcgan_results["epoch_times_s"]))
print(f"Training time: {dcgan_results['training_time_s']:.1f} s")
print(f"Mean epoch time: {dcgan_mean_epoch_time:.1f} s")
print(f"Checkpoint: {dcgan_results['checkpoint_path']}")

### 13.1. Выбор checkpoint по development FID

Официальный test не используется для выбора модели.

Для каждого сохранённого checkpoint вычисляется FID на
development-наборе из 10 000 real images и 10 000 generated
images. Для каждой архитектуры выбирается checkpoint с минимальным
development FID. После выбора этот checkpoint загружается в
генератор и используется для latent interpolation и финальной
test-оценки.


In [ ]:
def images_to_uint8(
    images,
):
    return (
        denormalize_images(
            images.detach().cpu()
        )
        .mul(255)
        .round()
        .to(torch.uint8)
    )


def save_image_batch(
    images,
    output_dir,
    start_index=0,
):
    output_dir = Path(
        output_dir
    )
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    uint8_images = (
        images_to_uint8(
            images
        )
    )

    for offset, image in enumerate(
        uint8_images
    ):
        image_array = (
            image.permute(
                1,
                2,
                0,
            )
            .contiguous()
            .numpy()
        )

        Image.fromarray(
            image_array
        ).save(
            output_dir
            / (
                "image_"
                f"{start_index + offset:06d}"
                ".png"
            )
        )

    return len(
        uint8_images
    )


def export_real_images(
    data_loader,
    output_dir,
    num_images,
):
    exported = 0

    for images, _ in data_loader:
        remaining = (
            num_images
            - exported
        )

        if remaining <= 0:
            break

        exported += save_image_batch(
            images[:remaining],
            output_dir,
            exported,
        )

    if exported != num_images:
        raise ValueError(
            f"Запрошено {num_images} "
            "real images, доступно "
            f"{exported}"
        )

    return exported


def export_generated_images(
    generator,
    latent_shape,
    output_dir,
    num_images,
    batch_size=BATCH_SIZE,
    seed=SEED,
    device=DEVICE,
):
    was_training = (
        generator.training
    )
    generator.eval()

    exported = 0
    noise_generator = (
        torch.Generator()
        .manual_seed(seed)
    )

    try:
        with torch.no_grad():
            while (
                exported
                < num_images
            ):
                current_batch = min(
                    batch_size,
                    num_images - exported,
                )

                noise = torch.randn(
                    current_batch,
                    *latent_shape,
                    generator=(
                        noise_generator
                    ),
                ).to(device)

                generated = generator(
                    noise
                )

                exported += (
                    save_image_batch(
                        generated,
                        output_dir,
                        exported,
                    )
                )
    finally:
        generator.train(
            was_training
        )

    return exported


def calculate_fid_only(
    generated_dir,
    real_dir,
    use_cuda,
):
    metrics = (
        torch_fidelity
        .calculate_metrics(
            input1=str(
                generated_dir
            ),
            input2=str(
                real_dir
            ),
            cuda=use_cuda,
            isc=False,
            fid=True,
            cache=False,
            rng_seed=SEED,
            verbose=False,
        )
    )

    return float(
        metrics[
            "frechet_inception_distance"
        ]
    )


def checkpoint_sha256(
    path,
):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def load_generator_checkpoint(
    generator,
    checkpoint_path,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    generator.load_state_dict(
        checkpoint[
            "generator_state_dict"
        ]
    )

    return checkpoint


def select_checkpoint_by_dev_fid(
    model_name,
    generator,
    latent_shape,
    checkpoint_paths,
    development_real_dir,
):
    rows = []

    with tempfile.TemporaryDirectory(
        prefix=(
            "dev_checkpoint_"
            + model_name.lower()
            .replace(" ", "_")
            + "_"
        ),
        dir=RUNTIME_ROOT,
    ) as temp_dir:
        temp_root = Path(
            temp_dir
        )

        for checkpoint_path in (
            checkpoint_paths
        ):
            checkpoint = (
                load_generator_checkpoint(
                    generator,
                    checkpoint_path,
                )
            )

            epoch = int(
                checkpoint["epoch"]
            )

            generated_dir = (
                temp_root
                / f"epoch_{epoch:03d}"
            )

            export_generated_images(
                generator=generator,
                latent_shape=(
                    latent_shape
                ),
                output_dir=(
                    generated_dir
                ),
                num_images=(
                    DEVELOPMENT_EVAL_IMAGES
                ),
                seed=SEED + 50,
                device=DEVICE,
            )

            fid_value = (
                calculate_fid_only(
                    generated_dir,
                    development_real_dir,
                    use_cuda=(
                        DEVICE.type
                        == "cuda"
                    ),
                )
            )

            rows.append(
                {
                    "Model": model_name,
                    "Epoch": epoch,
                    "Development FID": (
                        fid_value
                    ),
                    "Checkpoint": str(
                        checkpoint_path
                    ),
                    "Checkpoint SHA256": (
                        checkpoint_sha256(
                            checkpoint_path
                        )
                    ),
                }
            )

    table = pd.DataFrame(
        rows
    ).sort_values(
        "Development FID"
    )

    best_row = table.iloc[0]
    best_path = Path(
        best_row["Checkpoint"]
    )

    load_generator_checkpoint(
        generator,
        best_path,
    )

    return (
        best_path,
        table.reset_index(
            drop=True
        ),
    )


development_metric_root = (
    RUNTIME_ROOT
    / "development_metrics"
)
development_metric_root.mkdir(
    parents=True,
    exist_ok=True,
)

development_real_dir = (
    development_metric_root
    / "real"
)

if development_real_dir.exists():
    import shutil

    shutil.rmtree(
        development_real_dir
    )

export_real_images(
    development_loader,
    development_real_dir,
    DEVELOPMENT_EVAL_IMAGES,
)

(
    vanilla_selected_checkpoint,
    vanilla_dev_fid_df,
) = select_checkpoint_by_dev_fid(
    model_name="Vanilla GAN",
    generator=vanilla_generator,
    latent_shape=(LATENT_DIM,),
    checkpoint_paths=(
        vanilla_results[
            "checkpoint_paths"
        ]
    ),
    development_real_dir=(
        development_real_dir
    ),
)

(
    dcgan_selected_checkpoint,
    dcgan_dev_fid_df,
) = select_checkpoint_by_dev_fid(
    model_name="DCGAN",
    generator=dc_generator,
    latent_shape=(
        LATENT_DIM,
        1,
        1,
    ),
    checkpoint_paths=(
        dcgan_results[
            "checkpoint_paths"
        ]
    ),
    development_real_dir=(
        development_real_dir
    ),
)

development_selection_df = (
    pd.concat(
        [
            vanilla_dev_fid_df,
            dcgan_dev_fid_df,
        ],
        ignore_index=True,
    )
)

development_selection_df.to_csv(
    TABLE_ROOT
    / "development_checkpoint_selection.csv",
    index=False,
)

selected_checkpoint_by_model = {
    "Vanilla GAN": (
        vanilla_selected_checkpoint
    ),
    "DCGAN": (
        dcgan_selected_checkpoint
    ),
}

display(
    development_selection_df.style.format(
        {
            "Development FID": (
                "{:.3f}"
            )
        }
    )
)

print(
    "Vanilla selected:",
    vanilla_selected_checkpoint,
)
print(
    "DCGAN selected:",
    dcgan_selected_checkpoint,
)


### 14. Сравнение training dynamics и generated samples

Loss-кривые строятся для полного процесса обучения.
Финальные сравнительные grids ниже генерируются уже из checkpoint,
выбранных по development FID.


In [ ]:
def plot_loss_comparison(
    histories,
    output_path,
):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 4),
    )

    for (
        model_name,
        history,
    ) in histories.items():
        axes[0].plot(
            history["step"],
            history["loss_G"],
            label=model_name,
            alpha=0.85,
        )
        axes[1].plot(
            history["step"],
            history["loss_D"],
            label=model_name,
            alpha=0.85,
        )

    axes[0].set(
        title="Generator loss",
        xlabel="Training step",
        ylabel="BCE loss",
    )
    axes[1].set(
        title="Discriminator loss",
        xlabel="Training step",
        ylabel="BCE loss",
    )

    for axis in axes:
        axis.legend()
        axis.grid(
            alpha=0.25
        )

    fig.tight_layout()
    save_figure(
        fig,
        output_path,
    )
    plt.show()


def plot_selected_samples(
    output_path,
):
    vanilla_images = (
        generate_fixed_samples(
            vanilla_generator,
            fixed_noise_2d,
        )
    )
    dcgan_images = (
        generate_fixed_samples(
            dc_generator,
            fixed_noise_4d,
        )
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 6),
    )

    for (
        axis,
        model_name,
        images,
    ) in [
        (
            axes[0],
            "Vanilla GAN",
            vanilla_images,
        ),
        (
            axes[1],
            "DCGAN",
            dcgan_images,
        ),
    ]:
        grid = make_grid(
            denormalize_images(
                images
            ),
            nrow=8,
            padding=2,
        )

        axis.imshow(
            grid.permute(
                1,
                2,
                0,
            ).numpy()
        )
        axis.set_title(
            model_name
        )
        axis.axis("off")

    fig.tight_layout()
    save_figure(
        fig,
        output_path,
    )
    plt.show()


training_histories = {
    "Vanilla GAN": (
        vanilla_history
    ),
    "DCGAN": dcgan_history,
}

plot_loss_comparison(
    training_histories,
    FIGURE_ROOT
    / "gan_loss_comparison.png",
)

plot_selected_samples(
    FIGURE_ROOT
    / "gan_selected_samples.png"
)


### 14.1. Независимые сэмплы на новом шуме

Fixed-noise grids показывают динамику одних и тех же latent vectors.
Для дополнительной проверки разнообразия ниже создаются 64 новых
latent vectors, не использовавшихся в training snapshots.


In [ ]:
fresh_generator = (
    torch.Generator()
    .manual_seed(SEED + 777)
)

fresh_noise_2d = torch.randn(
    64,
    LATENT_DIM,
    generator=fresh_generator,
).to(DEVICE)

fresh_noise_4d = (
    fresh_noise_2d.view(
        64,
        LATENT_DIM,
        1,
        1,
    )
)

fresh_vanilla = (
    generate_fixed_samples(
        vanilla_generator,
        fresh_noise_2d,
    )
)
fresh_dcgan = (
    generate_fixed_samples(
        dc_generator,
        fresh_noise_4d,
    )
)

save_tensor_grid(
    fresh_vanilla,
    FIGURE_ROOT
    / "vanilla_fresh_noise.png",
    "Vanilla GAN: fresh noise",
    nrow=8,
)

save_tensor_grid(
    fresh_dcgan,
    FIGURE_ROOT
    / "dcgan_fresh_noise.png",
    "DCGAN: fresh noise",
    nrow=8,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 6),
)

for (
    axis,
    title,
    images,
) in [
    (
        axes[0],
        "Vanilla GAN",
        fresh_vanilla,
    ),
    (
        axes[1],
        "DCGAN",
        fresh_dcgan,
    ),
]:
    grid = make_grid(
        denormalize_images(
            images
        ),
        nrow=8,
        padding=2,
    )
    axis.imshow(
        grid.permute(
            1,
            2,
            0,
        ).numpy()
    )
    axis.set_title(
        title
    )
    axis.axis("off")

fig.tight_layout()
plt.show()


При просмотре сеток нужно оценить резкость границ, узнаваемость глобальных форм, цветовое разнообразие и долю повторяющихся образцов. Mode collapse можно отметить только при фактически наблюдаемом повторении малого числа шаблонов; по одному значению loss такой вывод делать нельзя. Если быстрый запуск создаёт шумоподобные изображения, это ожидаемо и не характеризует потенциал архитектуры после полного обучения.

### 15. Latent space interpolation

Между двумя latent vectors $z_1$ и $z_2$ строится линейный путь

$$z(\alpha)=(1-\alpha)z_1+\alpha z_2, \qquad \alpha \in [0, 1].$$

Десять равномерных значений $\alpha$ включают обе конечные точки. Для честного визуального сопоставления Vanilla GAN и DCGAN используют одинаковые численные значения latent path; для DCGAN добавляются только две пространственные размерности `1×1`. Если соседние изображения изменяются постепенно, это указывает, что генератор сформировал структурированное, а не полностью разорванное latent space.

In [ ]:
def interpolate_latent(start, end, steps=10):
    if start.shape != end.shape:
        raise ValueError("start и end должны иметь одинаковую форму")
    if steps < 2:
        raise ValueError("Для интерполяции требуется минимум два шага")
    alpha_shape = (steps,) + (1,) * start.ndim
    alphas = torch.linspace(
        0.0, 1.0, steps, device=start.device, dtype=start.dtype
    ).reshape(alpha_shape)
    return (1.0 - alphas) * start.unsqueeze(0) + alphas * end.unsqueeze(0)


def plot_latent_interpolation(images, title, output_path):
    display_images = denormalize_images(images)
    grid = make_grid(display_images, nrow=len(display_images), padding=2)
    fig, ax = plt.subplots(figsize=(16, 2.5))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    save_figure(fig, output_path)
    plt.show()

In [ ]:
INTERPOLATION_STEPS = 10
interpolation_generator = torch.Generator().manual_seed(SEED + 1)
latent_start = torch.randn(LATENT_DIM, generator=interpolation_generator).to(DEVICE)
latent_end = torch.randn(LATENT_DIM, generator=interpolation_generator).to(DEVICE)
vanilla_latent_path = interpolate_latent(
    latent_start, latent_end, steps=INTERPOLATION_STEPS
)
dcgan_latent_path = vanilla_latent_path.view(
    INTERPOLATION_STEPS, LATENT_DIM, 1, 1
)

assert torch.allclose(vanilla_latent_path[0], latent_start)
assert torch.allclose(vanilla_latent_path[-1], latent_end)
assert tuple(vanilla_latent_path.shape) == (INTERPOLATION_STEPS, LATENT_DIM)
assert tuple(dcgan_latent_path.shape) == (INTERPOLATION_STEPS, LATENT_DIM, 1, 1)

vanilla_interpolation_images = generate_fixed_samples(
    vanilla_generator, vanilla_latent_path
)
dcgan_interpolation_images = generate_fixed_samples(dc_generator, dcgan_latent_path)
plot_latent_interpolation(
    vanilla_interpolation_images,
    "Vanilla GAN: latent interpolation",
    FIGURE_ROOT / "vanilla_latent_interpolation.png",
)
plot_latent_interpolation(
    dcgan_interpolation_images,
    "DCGAN: latent interpolation",
    FIGURE_ROOT / "dcgan_latent_interpolation.png",
)

Интерпретация должна основываться на фактически отображённых рядах. Плавное изменение формы, цвета или текстуры между соседними кадрами свидетельствует о непрерывности изученного представления. Резкие скачки, почти одинаковые изображения на всём пути или повторение одного шаблона могут указывать на слабую организацию latent space либо mode collapse, однако такой вывод нельзя делать без наблюдаемого результата полного запуска.

### 16. FID и Inception Score

Финальная оценка выполняется **только после выбора checkpoint по
development**.

Для каждой модели:

- используется официальный CIFAR-10 test из 10 000 real images;
- независимо генерируются 10 000 fake images;
- применяется одна версия `torch-fidelity`;
- preprocessing одинаков: `[-1, 1] → uint8 [0, 255] → RGB PNG`;
- `IS splits = 10`;
- сохраняются raw metrics, seed, параметры протокола,
  checkpoint SHA-256 и число exact duplicates.

Test не используется для выбора checkpoint.


In [ ]:
def calculate_fid_and_is(
    generated_dir,
    real_dir,
    use_cuda,
):
    raw_metrics = (
        torch_fidelity
        .calculate_metrics(
            input1=str(
                generated_dir
            ),
            input2=str(
                real_dir
            ),
            cuda=use_cuda,
            isc=True,
            fid=True,
            isc_splits=IS_SPLITS,
            samples_shuffle=False,
            cache=False,
            rng_seed=SEED,
            verbose=True,
        )
    )

    summary = {
        "FID ↓": float(
            raw_metrics[
                "frechet_inception_distance"
            ]
        ),
        "Inception Score ↑": float(
            raw_metrics[
                "inception_score_mean"
            ]
        ),
        "IS std": float(
            raw_metrics[
                "inception_score_std"
            ]
        ),
    }

    serializable_raw = {
        key: (
            float(value)
            if isinstance(
                value,
                (int, float, np.number),
            )
            else str(value)
        )
        for key, value
        in raw_metrics.items()
    }

    return (
        summary,
        serializable_raw,
    )


def exact_duplicate_count(
    image_dir,
):
    hashes = []

    for path in sorted(
        Path(image_dir).glob("*.png")
    ):
        hashes.append(
            hashlib.sha256(
                path.read_bytes()
            ).hexdigest()
        )

    return (
        len(hashes)
        - len(set(hashes))
    )


In [ ]:
evaluation_image_count = (
    FAST_EVAL_IMAGES
    if FAST_MODE
    else FULL_EVAL_IMAGES
)

evaluation_label = (
    "quick/debug"
    if FAST_MODE
    else "full/final"
)

if not FAST_MODE:
    assert IS_KAGGLE
    assert DEVICE.type == "cuda"
    assert evaluation_image_count == 10_000
    assert len(test_dataset) == 10_000

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

metric_rows = []

with tempfile.TemporaryDirectory(
    prefix="fid_is_",
    dir=RUNTIME_ROOT,
) as temp_dir:
    metric_root = Path(
        temp_dir
    )
    real_metric_dir = (
        metric_root
        / "real"
    )

    real_count = export_real_images(
        test_loader,
        real_metric_dir,
        evaluation_image_count,
    )

    assert (
        real_count
        == evaluation_image_count
    )

    metric_model_specs = [
        (
            "Vanilla GAN",
            vanilla_generator,
            (LATENT_DIM,),
        ),
        (
            "DCGAN",
            dc_generator,
            (
                LATENT_DIM,
                1,
                1,
            ),
        ),
    ]

    for (
        model_name,
        generator,
        latent_shape,
    ) in metric_model_specs:
        generated_metric_dir = (
            metric_root
            / model_name.lower()
            .replace(" ", "_")
        )

        fake_count = (
            export_generated_images(
                generator=generator,
                latent_shape=(
                    latent_shape
                ),
                output_dir=(
                    generated_metric_dir
                ),
                num_images=(
                    evaluation_image_count
                ),
                seed=SEED + 100,
                device=DEVICE,
            )
        )

        assert (
            fake_count
            == evaluation_image_count
        )

        (
            metrics,
            raw_metrics,
        ) = calculate_fid_and_is(
            generated_metric_dir,
            real_metric_dir,
            use_cuda=(
                DEVICE.type
                == "cuda"
            ),
        )

        checkpoint_path = (
            selected_checkpoint_by_model[
                model_name
            ]
        )

        checkpoint_hash = (
            checkpoint_sha256(
                checkpoint_path
            )
        )

        duplicates = (
            exact_duplicate_count(
                generated_metric_dir
            )
        )

        protocol = {
            "run_id": RUN_ID,
            "model": model_name,
            "evaluation": (
                evaluation_label
            ),
            "real_source": (
                "official CIFAR-10 test"
            ),
            "real_images": (
                real_count
            ),
            "generated_images": (
                fake_count
            ),
            "generated_seed": (
                SEED + 100
            ),
            "preprocessing": (
                "[-1, 1] -> "
                "uint8 [0, 255] -> RGB PNG"
            ),
            "torch_fidelity_version": (
                installed_version(
                    "torch-fidelity"
                )
            ),
            "is_splits": IS_SPLITS,
            "checkpoint": str(
                checkpoint_path
            ),
            "checkpoint_sha256": (
                checkpoint_hash
            ),
            "exact_duplicate_count": (
                duplicates
            ),
            "raw_metrics": raw_metrics,
        }

        protocol_path = (
            TABLE_ROOT
            / (
                model_name.lower()
                .replace(" ", "_")
                + "_test_metric_protocol.json"
            )
        )

        protocol_path.write_text(
            json.dumps(
                protocol,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        metric_rows.append(
            {
                "Model": model_name,
                "Evaluation": (
                    evaluation_label
                ),
                "Images": (
                    evaluation_image_count
                ),
                "IS splits": IS_SPLITS,
                "Checkpoint SHA256": (
                    checkpoint_hash
                ),
                "Exact duplicates": (
                    duplicates
                ),
                **metrics,
            }
        )

gan_metrics_df = pd.DataFrame(
    metric_rows
)

gan_metrics_df.to_csv(
    TABLE_ROOT
    / "gan_metrics.csv",
    index=False,
)

gan_metrics_df.to_json(
    TABLE_ROOT
    / "gan_metrics.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

display(
    gan_metrics_df.style.format(
        {
            "FID ↓": "{:.3f}",
            "Inception Score ↑": (
                "{:.3f}"
            ),
            "IS std": "{:.3f}",
        }
    )
)

if FAST_MODE:
    print(
        "ВНИМАНИЕ: quick/debug FID и IS "
        "не являются финальным результатом."
    )


Таблица выше создаётся только из значений, возвращённых `torch-fidelity` в текущем запуске. Меньший FID указывает на более близкое к CIFAR-10 распределение признаков, а больший IS — на более уверенные и разнообразные предсказания Inception. Метрики чувствительны к числу samples и preprocessing, поэтому Vanilla GAN и DCGAN оцениваются одним кодом, на одинаковом количестве изображений и с одинаковым seed. Quick/debug и full/final результаты нельзя смешивать в одном выводе.

### 17. Сравнение Vanilla GAN и DCGAN

Финальная таблица объединяет метрики качества, вычислительное время и размер моделей из runtime variables. Строки `quick/debug` отражают только технический запуск; для итогового отчёта Notebook необходимо выполнить с `RUN_MODE = "full"`.

In [ ]:
training_summary_df = pd.DataFrame(
    [
        {
            "Model": "Vanilla GAN",
            "Training time (s)": (
                vanilla_results[
                    "training_time_s"
                ]
            ),
            "Epoch time (s)": (
                vanilla_mean_epoch_time
            ),
            "G params": count_parameters(
                vanilla_generator
            ),
            "D params": count_parameters(
                vanilla_discriminator
            ),
        },
        {
            "Model": "DCGAN",
            "Training time (s)": (
                dcgan_results[
                    "training_time_s"
                ]
            ),
            "Epoch time (s)": (
                dcgan_mean_epoch_time
            ),
            "G params": count_parameters(
                dc_generator
            ),
            "D params": count_parameters(
                dc_discriminator
            ),
        },
    ]
)

training_summary_df[
    "Total params"
] = (
    training_summary_df["G params"]
    + training_summary_df["D params"]
)

comparison_df = (
    training_summary_df.merge(
        gan_metrics_df,
        on="Model",
        how="inner",
        validate="one_to_one",
    )
)

comparison_columns = [
    "Model",
    "Evaluation",
    "Images",
    "IS splits",
    "FID ↓",
    "Inception Score ↑",
    "IS std",
    "Exact duplicates",
    "Training time (s)",
    "Epoch time (s)",
    "G params",
    "D params",
    "Total params",
    "Checkpoint SHA256",
]

comparison_df = (
    comparison_df[
        comparison_columns
    ]
)

comparison_df.to_csv(
    TABLE_ROOT
    / "gan_comparison.csv",
    index=False,
)

comparison_df.to_json(
    TABLE_ROOT
    / "gan_comparison.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

display(
    comparison_df.style.format(
        {
            "FID ↓": "{:.3f}",
            "Inception Score ↑": (
                "{:.3f}"
            ),
            "IS std": "{:.3f}",
            "Training time (s)": (
                "{:.1f}"
            ),
            "Epoch time (s)": (
                "{:.1f}"
            ),
            "G params": "{:,}",
            "D params": "{:,}",
            "Total params": "{:,}",
        }
    )
)


### 18. Анализ результатов и ограничений

При интерпретации учитываются одновременно:

- `loss_G` и `loss_D` как диагностика устойчивости обучения;
- fixed-noise и fresh-noise grids;
- latent interpolation;
- FID и Inception Score;
- число точных дубликатов среди 10 000 generated images.

Полный mode collapse нельзя устанавливать только по loss.
Его признаки ищутся по повторяющимся сэмплам, низкому разнообразию
fresh-noise grids и числу exact duplicates.

Ограничения эксперимента:

- CIFAR-10 имеет разрешение 32×32;
- модели безусловные, class labels не управляют генерацией;
- выполнен один основной seed;
- FID/IS чувствительны к preprocessing и числу samples;
- сравниваются только Vanilla GAN и DCGAN.

WGAN-GP, Conditional GAN, StyleGAN2-ADA, VAE/DDPM и
Precision/Recall for Generative Models являются дополнительными
расширениями и не входят в обязательную базовую часть.


### 19. Вывод

Следующая ячейка формирует вывод из фактических значений текущего запуска. Она не содержит заранее заданных FID, IS, времени или победителя.

In [ ]:
best_fid_row = comparison_df.loc[
    comparison_df["FID ↓"].idxmin()
]

best_is_row = comparison_df.loc[
    comparison_df[
        "Inception Score ↑"
    ].idxmax()
]

fastest_training_row = (
    comparison_df.loc[
        comparison_df[
            "Training time (s)"
        ].idxmin()
    ]
)

smallest_model_row = (
    comparison_df.loc[
        comparison_df[
            "Total params"
        ].idxmin()
    ]
)

if (
    best_fid_row["Model"]
    == best_is_row["Model"]
):
    metric_agreement = (
        "Обе метрики качества выделяют "
        f"{best_fid_row['Model']}."
    )
else:
    metric_agreement = (
        "Метрики расходятся: лучший FID у "
        f"{best_fid_row['Model']}, "
        "а лучший Inception Score у "
        f"{best_is_row['Model']}."
    )

result_scope = (
    "Предварительный debug-вывод; "
    "он не является итогом лабораторной."
    if FAST_MODE
    else (
        "Финальный вывод по Kaggle GPU "
        "на 10 000 real/generated images."
    )
)

conclusion_text = (
    f"**{result_scope}**\n\n"
    "- Минимальный FID: "
    f"**{best_fid_row['Model']}** — "
    f"{best_fid_row['FID ↓']:.3f}.\n"
    "- Максимальный Inception Score: "
    f"**{best_is_row['Model']}** — "
    f"{best_is_row['Inception Score ↑']:.3f}.\n"
    "- Меньшее время обучения: "
    f"**{fastest_training_row['Model']}** — "
    f"{fastest_training_row['Training time (s)']:.1f} s.\n"
    "- Меньше обучаемых параметров: "
    f"**{smallest_model_row['Model']}** — "
    f"{int(smallest_model_row['Total params']):,}.\n"
    f"- {metric_agreement}\n\n"
    "GAN losses используются для диагностики, "
    "а итог качества определяется совместно "
    "по FID, IS и визуальным результатам."
)

CONCLUSION_PATH = (
    ARTIFACT_ROOT
    / "conclusion.md"
)

CONCLUSION_PATH.write_text(
    conclusion_text + "\n",
    encoding="utf-8",
)

display(
    Markdown(
        conclusion_text
    )
)

print(
    "Итоговые артефакты:",
    ARTIFACT_ROOT,
)


### 20. Проверка требований

В полном запуске должны быть выполнены следующие условия:

- обучены Vanilla GAN и DCGAN;
- full-run выполнен на Kaggle GPU;
- изображения нормализованы в `[-1, 1]`;
- сохранены train/development/test manifests;
- логируются `loss_G` и `loss_D`;
- checkpoint и fixed-noise snapshots сохраняются регулярно;
- checkpoint выбирается по development FID;
- официальный test не используется при выборе checkpoint;
- показаны 8×8 grids и fresh-noise samples;
- выполнена latent interpolation;
- для каждой модели FID рассчитан на 10 000 real и 10 000 fake;
- для обеих моделей рассчитан Inception Score;
- сохранены raw metric results, seed, IS splits и checkpoint SHA-256;
- выполнена проверка exact duplicates как одного из индикаторов
  mode collapse;
- ограничения результата явно описаны.


In [ ]:
if not FAST_MODE:
    assert IS_KAGGLE
    assert DEVICE.type == "cuda"
    assert len(train_dataset) == 40_000
    assert len(development_dataset) == 10_000
    assert len(test_dataset) == 10_000

    assert set(
        gan_metrics_df["Model"]
    ) == {
        "Vanilla GAN",
        "DCGAN",
    }

    assert (
        gan_metrics_df["Images"]
        == 10_000
    ).all()

    assert (
        gan_metrics_df[
            "IS splits"
        ]
        == IS_SPLITS
    ).all()

    assert all(
        Path(path).is_file()
        for path in (
            vanilla_selected_checkpoint,
            dcgan_selected_checkpoint,
        )
    )

    assert (
        vanilla_results[
            "checkpoint_paths"
        ]
    )

    assert (
        dcgan_results[
            "checkpoint_paths"
        ]
    )

    print(
        "Обязательные проверки "
        "полного протокола пройдены."
    )
else:
    print(
        "Debug-режим: финальная "
        "приёмка не выполняется."
    )
